# Convert (country, lat, lon) mask to (lat, lon) mask with country IDs

Converting DataArray that contains 0.1ºx0.1º gridded masks for all 204 countries (McDuffie et al. 2021, Zenodo, https://doi.org/10.5281/zenodo.4642700) to a single `(lat, lon)` array where each gridpoint holds the country's integer ID (matching the JSON created in `0__Save_locationID_to_countryname.ipynb`), and unassigned/ocean gridpoints are `NaN`.

In [1]:
import os
import json
import numpy as np
import xarray as xr
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
JSON_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")

In [3]:
# Load lookup json
json_file = "country_location_ids.json"
json_path = os.path.join(JSON_DIR, json_file)
with open(json_path) as f:
    name_to_id = json.load(f)

In [4]:
# Load country masks
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
mask = xr.open_dataarray(mask_path)

### Build the (lat, lon) ID mask

For each gridpoint, we find the position along `country` where the binary mask is 1
(first match wins if there's ever more than one), look up that position's country
name, translate it to the JSON's integer ID, and place that ID at the gridpoint.
Gridpoints that are all-zero across every country (ocean / unassigned) become `NaN`.


In [5]:
binary = mask.values  # shape: (n_country, n_lat, n_lon)
n_country, n_lat, n_lon = binary.shape

# "First match wins": argmax on a 0/1 array returns the index of the first 1
# if any 1 exists, else 0. We separately detect true all-zero points.
has_any = binary.any(axis=0)                # (lat, lon) bool: True if assigned to >=1 country
first_match_idx = np.argmax(binary, axis=0)  # (lat, lon) int: index along country dim

# Sanity check: flag (don't silently ignore) any gridpoint assigned to >1 country
overlap_counts = binary.sum(axis=0)
n_overlaps = int((overlap_counts > 1).sum())
if n_overlaps > 0:
    print(f"WARNING: {n_overlaps} gridpoint(s) are 1 in more than one country's layer. "
          "Using the first match along the country dimension for these, as requested.")
else:
    print("No overlapping gridpoints found (each assigned point belongs to exactly one country).")


The grid has 1800 × 3600 = 6,480,000 total gridpoints. 53 of those overlapping accounts for ≈ 0.0008% — i.e. about 1 in every 122,000 gridpoints.  

This can be ignored.

In [6]:
# Translate country-dimension position -> country name -> JSON ID
country_coord_values = [str(v) for v in mask["country"].values]
country_names_arr = np.array(country_coord_values)

# Build position -> ID lookup
pos_to_id = np.full(n_country, np.nan)
unmapped_names = []
for pos, name in enumerate(country_names_arr):
    if name in name_to_id:
        pos_to_id[pos] = name_to_id[name]
    else:
        unmapped_names.append(name)

if unmapped_names:
    print(f"NOTE: {len(unmapped_names)} countries in the mask have no JSON ID and will "
          f"map to NaN wherever they're the first match: {unmapped_names}")

# Vectorized lookup: for each gridpoint, get the ID for its first-match country position
id_grid = pos_to_id[first_match_idx]   # (lat, lon) float, NaN where pos_to_id was NaN

# Overwrite truly unassigned (all-zero) gridpoints with NaN
id_grid = np.where(has_any, id_grid, np.nan)

print("Output shape:", id_grid.shape)
print("Unique IDs present (excluding NaN):", len(np.unique(id_grid[~np.isnan(id_grid)]).astype(int)))
print(f"Fraction of gridpoints unassigned (NaN): {np.isnan(id_grid).mean():.2%}")

country_id_mask = xr.DataArray(
    id_grid,
    dims=("lat", "lon"),
    coords={"lat": mask["lat"].values, "lon": mask["lon"].values},
    name="country_id",
    attrs={
        "description": "Country ID per gridpoint, derived from binary (country, lat, lon) mask",
        "id_source": "country_location_ids.json",
        "nodata_value": "NaN (ocean / unassigned)",
    },
)

out_file = "country_id_mask.nc"
out_path = os.path.join(MASKS_DIR, out_file)

print(f"Saving to {out_path}")
country_id_mask.to_netcdf(out_path)

print("All processing complete.")

Output shape: (1800, 3600)
Unique IDs present (excluding NaN): 199
Fraction of gridpoints unassigned (NaN): 76.33%
Saving to /glade/work/awells/workflow/BMR/masks/country/country_id_mask.nc
All processing complete.


### Checks

In [7]:
# 1. Are IDs in the JSON unique? (Duplicate IDs would shrink the unique count)
print("JSON IDs:", len(name_to_id), "| unique IDs:", len(set(name_to_id.values())))

# 2. Which countries are missing from the output?
present_ids = set(np.unique(id_grid[~np.isnan(id_grid)]).astype(int))
missing = {n: i for n, i in name_to_id.items() if i not in present_ids}
print(f"\nMissing from output ({len(missing)}): {missing}\n")

# 3. For each, why?
for name, cid in missing.items():
    pos = country_coord_values.index(name)
    layer = binary[pos].astype(bool)
    n_cells = int(layer.sum())
    lost = layer & (first_match_idx != pos)          # cells it owns but another country won
    winners = {country_coord_values[w] for w in np.unique(first_match_idx[lost])}
    print(f"{name} (ID {cid}): {n_cells} cells in its layer, "
          f"{int(lost.sum())} lost to tiebreak -> winners: {winners or '-'}")

JSON IDs: 204 | unique IDs: 204

Missing from output (5): {'Monaco': 367, 'Marshall Islands': 24, 'Nauru': 369, 'Tokelau': 413, 'Tuvalu': 416}

Monaco (ID 367): 0 cells in its layer, 0 lost to tiebreak -> winners: -
Marshall Islands (ID 24): 0 cells in its layer, 0 lost to tiebreak -> winners: -
Nauru (ID 369): 0 cells in its layer, 0 lost to tiebreak -> winners: -
Tokelau (ID 413): 0 cells in its layer, 0 lost to tiebreak -> winners: -
Tuvalu (ID 416): 0 cells in its layer, 0 lost to tiebreak -> winners: -
